<a href="https://colab.research.google.com/github/wonzzae/WJ-Archive/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D%20%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install fastapi uvicorn gradio beautifulsoup4 requests nest_asyncio

In [20]:
import requests
from bs4 import BeautifulSoup

def crawl_quotes():
    url = "http://quotes.toscrape.com/"
    res = requests.get(url)
    soup = BeautifulSoup(res.text, "html.parser")

    data = []
    for q in soup.select(".quote")[:20]:
        text = q.select_one(".text").get_text()
        author = q.select_one(".author").get_text()
        tags = [t.get_text() for t in q.select(".tag")]

        data.append((text, author, ", ".join(tags)))

    return data

In [21]:
import sqlite3

conn = sqlite3.connect("quotes.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS quotes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    text TEXT,
    author TEXT,
    tags TEXT
)
""")

data = crawl_quotes()

cur.execute("DELETE FROM quotes")  # 초기화
for d in data:
    cur.execute("INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)", d)

conn.commit()
conn.close()

print("저장 완료:", len(data))

저장 완료: 10


In [22]:
from fastapi import FastAPI
import sqlite3
import random

app = FastAPI()

@app.get("/quotes")
def get_quotes():
    conn = sqlite3.connect("quotes.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM quotes")
    data = cur.fetchall()
    conn.close()
    return data

@app.get("/random")
def random_quote():
    conn = sqlite3.connect("quotes.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM quotes")
    data = cur.fetchall()
    conn.close()
    return random.choice(data)

In [23]:
import gradio as gr
import requests

def get_data():
    return requests.get("http://localhost:8000/quotes").json()

def get_random():
    return requests.get("http://localhost:8000/random").json()

with gr.Blocks() as demo:
    gr.Markdown("# 📚 Quotes Dashboard")

    btn1 = gr.Button("전체 조회")
    btn2 = gr.Button("랜덤 명언")

    out = gr.JSON()

    btn1.click(get_data, outputs=out)
    btn2.click(get_random, outputs=out)

In [24]:
from gradio import mount_gradio_app
app = mount_gradio_app(app, demo, path="/")

new /


In [29]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import threading
import time

# 기존 터널 제거
ngrok.kill()

# 토큰 입력
from getpass import getpass
token = getpass("ngrok 토큰 입력: ")
ngrok.set_auth_token(token)

# 서버 실행
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

# 서버 준비 시간
time.sleep(3)

# ngrok 연결
public_url = ngrok.connect(8000)
print("🔥 ngrok URL:", public_url)

ngrok 토큰 입력: ··········


INFO:     Started server process [13446]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔥 ngrok URL: NgrokTunnel: "https://pungent-gentile-hydrogen.ngrok-free.dev" -> "http://localhost:8000"


이 밑에 URL 을 새창에서 실행


https://pungent-gentile-hydrogen.ngrok-free.dev/docs

https://pungent-gentile-hydrogen.ngrok-free.dev/gradio


토큰 : 3DLkLAprOS0Nvz1nbA8JmwSm8n1_2W1LEUgATFFVrTMbS9ucc

